# Python Foundations for Data Engineering: Full Overview
### Instructor Demo (Charter Oak Mutual claims) | Python 3.13

This notebook covers every concept the drills, Advanced Track, and both mini
projects practice, from the **basics** to **modern Python 3.7+ features**,
organized the way a data engineer works:

> **Ingest -> Inspect -> Transform -> Analyze -> Output**

Teach it top to bottom in VS Code, then move to `demo_pipeline.py` to show the
script form your drills and homework use.

**Map of this notebook**
1. Basics: types, strings, collections, control flow, functions, exceptions
   (including a custom exception and a retry loop)
2. Data I/O and aggregation tools: text, CSV, JSON, SQLite, `collections`
   (`Counter`/`defaultdict`), `datetime`
3. Modern Python (3.7+): comprehensions, lambdas/map/filter, generators,
   `itertools`, walrus, match/case, classes, dataclasses + type hints,
   decorators
4. Put it together: a small pipeline that is Mini Project 1 in miniature, a
   preview of Mini Project 2, then notebook -> script

See the coverage map in the next cell for exactly which drill, Advanced Track
item, or mini project each section demos live.

## Coverage map: what this demo shows, and where it lives in the day

| This demo section | Drill / item it demos |
|---|---|
| Variables and types, f-strings, strings | 01-Stu_Variables |
| Control flow: conditionals | 02-Stu_Conditionals, 03-Stu_Triage (the rank-comparison pattern) |
| Control flow: loops | 04-Stu_Loops, 07-Stu_Iterate_Lists |
| Collections: list, tuple, set, dict | 05-Stu_Lists, 06-Stu_Dictionaries |
| Functions | 08-Stu_Functions |
| Exceptions, custom exception, retry loop | 16-Stu_Error_Handling |
| Text files | 11-Stu_File_IO |
| CSV | 12-Stu_CSV_IO |
| JSON | 13-Stu_JSON_IO, 09-Stu_Nesting (the nested shape) |
| SQLite | 15-Stu_SQLite |
| `collections`: Counter, defaultdict | 17-Stu_Collections |
| `datetime` | 18-Stu_Datetime |
| Comprehensions | A1-Stu_Comprehensions (promoted into core drill 14 too) |
| Lambdas, map, filter | 14-Stu_Lambdas_Map_Filter |
| Generators | stretch tiers in 07 and 09 |
| `itertools` | A4-Stu_Itertools |
| Walrus operator | stretch tiers in 07 and 09 |
| match/case | A2-Stu_Pattern_Matching |
| Classes | 10-Stu_Classes |
| Dataclasses + type hints | A3-Stu_Classes_to_Dataclasses |
| Decorators | A5-Stu_Decorator |
| The aggregate pipeline in Part 4 | Mini Project 1: `Activity_1_Mini_Capstone_Claims_Intake/` (Core + Challenge tiers) |
| The Mini Project callout after Part 4 | Mini Project 2: `Activity_2_No_Show_Analysis/` (previewed, not solved live) |

Every drill, every Advanced Track item, and both mini projects are represented
somewhere in this notebook. If a table students never build (the Stretch-tier
walrus/generator work) is not obvious in a section above, it means the pattern
is demoed inline within that section's code rather than getting its own
heading.

# Part 1: The Basics

## Variables and types
Everything is an object with a type. Know your types.

In [ ]:
claim_id = "CLM-D01"   # str
reserve = 5000.0        # float
open_count = 3          # int
is_open = True          # bool
adjuster = None         # NoneType (absence of a value)

for value in (claim_id, reserve, open_count, is_open, adjuster):
    print(f"{type(value).__name__:<8} -> {value}")

## f-strings and format specifiers
Format numbers for humans.

In [ ]:
reserve = 5000.0
claims = 3421
rate = 0.2012
print(f"${reserve:,.2f} reserve across {claims:,} claims")
print(f"no-show rate: {rate:.1%}")   # percent format

## Strings
Text cleaning is half of data engineering.

In [ ]:
raw = "  Auto, Property , Liability  "
parts = [p.strip().lower() for p in raw.split(",")]
print(parts)
print("auto".upper(), "PROPERTY".lower(), "liability".capitalize())
print("CLM-D01".startswith("CLM"), "CLM-D01"[-2:])

## Collections: list, tuple, set, dict
The four containers you reach for daily.

In [ ]:
# list: ordered, mutable
queue = ["CLM-1", "CLM-2", "CLM-3"]
queue.append("CLM-4")
print("list:", queue, "| first two:", queue[:2], "| len:", len(queue))

# tuple: ordered, immutable, great for fixed records and unpacking
record = ("CLM-1", "auto", 5000.0)
cid, ptype, res = record
print("tuple unpack:", cid, ptype, res)

# set: unique values, fast membership
types = {"auto", "property", "auto", "liability"}
print("set:", sorted(types), "| 'auto' in set:", "auto" in types)

# dict: key -> value
claim = {"claim_id": "CLM-1", "reserve": 5000.0}
claim["paid"] = 1200.0
print("dict.get with default:", claim.get("status", "open"))
for k, v in claim.items():
    print("   ", k, "=", v)

## Control flow: conditionals
`if / elif / else` with comparison and logical operators.

In [ ]:
reserve, manager_override = 22000, False
if reserve >= 20000 and not manager_override:
    tier = "severe"
elif reserve >= 5000:
    tier = "major"
else:
    tier = "minor"
print("tier:", tier)

## Control flow: loops
`for`, `range`, `enumerate`, `zip`, and `while`.

In [ ]:
amounts = [1200, 5000, 800]
for i, amount in enumerate(amounts, start=1):
    print(f"row {i}: {amount}")

for ptype, amount in zip(["auto", "property"], [1200, 5000]):
    print(ptype, amount)

n = 3
while n > 0:
    print("retry", n)
    n -= 1

## Functions
Name a calculation once; default arguments and docstrings.

In [ ]:
def loss_ratio(paid, reserve, round_to=1):
    """Paid divided by reserve, as a percentage."""
    if reserve == 0:
        return 0.0
    return round(paid / reserve * 100, round_to)

print(loss_ratio(1200, 5000), loss_ratio(0, 0), loss_ratio(1234, 5000, round_to=3))

## Exceptions
Real data is messy. Handle bad values instead of crashing, with an exception
specific enough to be useful. Then add a retry loop for flaky calls, the exact
pattern you will need for Day 3 API ingestion.

In [ ]:
def to_float(raw):
    try:
        return float(raw)
    except ValueError:
        return None
    finally:
        pass  # finally always runs; used for cleanup

print(to_float("18.5"), to_float("N/A"))

# A custom exception is more specific and more debuggable than a bare ValueError
class InvalidAmountError(ValueError):
    """Raised when an amount is missing or not a valid non-negative number."""

def parse_amount(raw):
    try:
        value = float(raw)
    except (TypeError, ValueError):
        raise InvalidAmountError(f"cannot parse amount: {raw!r}")
    if value < 0:
        raise InvalidAmountError(f"negative amount: {value}")
    return round(value, 2)

for raw in ["1200.50", "N/A", None, "-50"]:
    try:
        print("parsed:", parse_amount(raw))
    except InvalidAmountError as exc:
        print("rejected:", exc)

# PAUSE: what would a bare `except Exception: pass` hide here that catching
# InvalidAmountError specifically would not?

# A retry loop: the pattern you will build properly on Day 3 with httpx
import random

def flaky_fetch(claim_id):
    if random.random() < 0.5:
        raise ConnectionError("temporary network error")
    return {"claim_id": claim_id, "status": "ok"}

def fetch_with_retry(claim_id, max_tries=3):
    for attempt in range(1, max_tries + 1):
        try:
            return flaky_fetch(claim_id)
        except ConnectionError as exc:
            print(f"  attempt {attempt} failed: {exc}")
    raise RuntimeError(f"gave up on {claim_id} after {max_tries} tries")

try:
    print("retry result:", fetch_with_retry("CLM-D01"))
except RuntimeError as exc:
    print("exhausted:", exc)

# Part 2: Data I/O (ingest and output)
Data engineering starts by reading data and ends by writing it.

## Text files

In [ ]:
from pathlib import Path
Path("demo_note.txt").write_text("claims processed: 8\n")
print(Path("demo_note.txt").read_text().strip())
Path("demo_note.txt").unlink()  # cleanup

## CSV: ingest the claims (we reuse `claims` for the rest of the notebook)

In [ ]:
import csv
claims = []
with open("claims.csv", newline="") as f:
    for row in csv.DictReader(f):
        row["reserve"] = float(row["reserve"])
        row["paid"] = float(row["paid"])
        claims.append(row)
print(f"ingested {len(claims)} claims; first:", claims[0])

## JSON
The shape a REST API returns (Day 3). Serialize and parse.

In [ ]:
import json
payload = json.dumps(claims[0], indent=2)
print(payload)
parsed = json.loads(payload)
print("round-tripped:", parsed["claim_id"])

## SQLite
A real database with SQL, no install. `GROUP BY` here is the exact SQL shape
you will write in BigQuery next month; only the engine and the row count
change.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")   # in-memory db for the demo
cur = conn.cursor()
cur.execute("CREATE TABLE claims (id TEXT, ptype TEXT, reserve REAL, paid REAL)")
cur.executemany(
    "INSERT INTO claims VALUES (?, ?, ?, ?)",
    [(c["claim_id"], c["policy_type"], c["reserve"], c["paid"]) for c in claims],
)
conn.commit()

print("Reserve by policy type:")
for row in cur.execute("SELECT ptype, SUM(reserve) FROM claims GROUP BY ptype ORDER BY ptype"):
    print(" ", row)

print("\nLoss ratio by policy type, computed entirely in SQL:")
for row in cur.execute(
    "SELECT ptype, ROUND(SUM(paid) / SUM(reserve) * 100, 1) AS loss_ratio "
    "FROM claims GROUP BY ptype ORDER BY ptype"
):
    print(f"  {row[0]}: {row[1]}%")

conn.close()

# PAUSE: this GROUP BY / SUM / ORDER BY is the exact SQL shape you write in
# BigQuery in Week 3, just against millions of rows instead of five.

## The `collections` module: `Counter` and `defaultdict`
The thing you do constantly in data engineering is group and count. `Counter`
tallies occurrences; `defaultdict` gives every new key a starting value so you
skip the `setdefault` dance.

In [ ]:
from collections import Counter, defaultdict

counts = Counter(c["policy_type"] for c in claims)
print("Claim counts by type:", dict(counts))
print("Most common:", counts.most_common(1))

paid_by_type = defaultdict(float)
for c in claims:
    paid_by_type[c["policy_type"]] += c["paid"]
print("Total paid by type:", dict(paid_by_type))

# PAUSE: how would you write paid_by_type without defaultdict? (setdefault, or
# an "if key not in dict" check). Bridge to Day 4: pandas.groupby(...).sum()
# does this in one line; this is the manual version underneath it.

## `datetime`
Dates arrive as strings. Parse them once, then you can compare, sort, and
compute durations.

In [ ]:
from datetime import datetime

opened = {
    "CLM-D01": "2026-05-02",
    "CLM-D03": "2026-05-09",
    "CLM-D05": "2026-05-14",
}

parsed = {cid: datetime.strptime(d, "%Y-%m-%d") for cid, d in opened.items()}
oldest = min(parsed, key=parsed.get)
print("Oldest open claim:", oldest, parsed[oldest].strftime("%B %d, %Y"))

# How many days has each claim been open, as of a fixed "today"
today = datetime(2026, 6, 29)
for cid, d in sorted(parsed.items(), key=lambda kv: kv[1]):
    print(f"  {cid}: open {(today - d).days} days")

# PAUSE: subtracting two datetimes gives a timedelta, not an int; .days
# extracts the whole-day count. This is exactly how you compute SLA age.

# Part 3: Modern Python (3.7+)
The features that separate tutorial code from professional code.

## Comprehensions
Build a list, dict, or set in one readable line.

In [ ]:
open_ids = [c["claim_id"] for c in claims if c["status"] == "open"]
reserve_by_id = {c["claim_id"]: c["reserve"] for c in claims}
policy_types = {c["policy_type"] for c in claims}
print(open_ids)
print(sorted(policy_types))

## Lambdas, map, filter
The transform pattern you reuse in PySpark (Week 5).

In [ ]:
by_reserve = sorted(claims, key=lambda c: c["reserve"], reverse=True)
print("top 3 by reserve:", [c["claim_id"] for c in by_reserve][:3])
bumped = list(map(lambda c: round(c["reserve"] * 1.10, 2), claims))
open_only = list(filter(lambda c: c["status"] == "open", claims))
print("bumped:", bumped[:3], "| open:", len(open_only))

## Generators
Produce values lazily, one at a time (memory friendly for big data).

In [ ]:
def claims_over(rows, threshold):
    for c in rows:
        if c["reserve"] > threshold:
            yield c["claim_id"]

print("reserve over 10k:", list(claims_over(claims, 10000)))

## `itertools`
Built-in tools for working with iterators without loading everything into
memory at once. This is the Advanced Track A4 material.

In [ ]:
from itertools import chain, islice, accumulate, groupby

batch1 = claims[:2]
batch2 = claims[2:4]

# chain: merge two batches into one stream without building a new list
merged = list(chain(batch1, batch2))
print("Chained ids:", [c["claim_id"] for c in merged])

# islice: take the first 2 from a stream without materializing the rest
first_two = list(islice(chain(batch1, batch2), 2))
print("First two:", [c["claim_id"] for c in first_two])

# accumulate: running total
paids = [c["paid"] for c in merged]
print("Running total:", list(accumulate(paids)))

# groupby: MUST sort by the same key first, or groups fragment
by_type = sorted(claims, key=lambda c: c["policy_type"])
print("Grouped by policy type:")
for ptype, group in groupby(by_type, key=lambda c: c["policy_type"]):
    print(f"  {ptype}: {[c['claim_id'] for c in group]}")

# PAUSE: groupby groups CONSECUTIVE matching items only. Unsorted input
# silently produces fragmented, wrong groups. This is the #1 itertools bug.

## Walrus operator `:=` (3.8)
Assign and test in one expression.

In [ ]:
running = 0
for c in claims:
    if (running := running + c["paid"]) > 30000:
        print(f"cumulative paid crossed 30k at {c['claim_id']} (= {running})")
        break

## match/case (3.10)
Match the shape of data, not just one value.

In [ ]:
def route(claim):
    match claim:
        case {"status": "denied"}:
            return "closed"
        case {"policy_type": "auto", "reserve": r} if r > 4000:
            return "auto-review"
        case {"status": "open"}:
            return "standard"
        case _:
            return "manual"

for c in claims[:4]:
    print(c["claim_id"], "->", route(c))

## Classes
Bundle data and behavior. Use a class for a thing with state; a function for a pure calculation.

In [ ]:
class Claim:
    def __init__(self, claim_id, reserve, paid):
        self.claim_id = claim_id
        self.reserve = reserve
        self.paid = paid

    def burn(self):
        return round(self.paid / self.reserve * 100, 1)

c = Claim("CLM-1", 5000.0, 1200.0)
print(c.claim_id, "burn:", c.burn(), "%")

## Dataclasses (3.7) + type hints
Same class, less boilerplate. The dataclass writes `__init__`, `__repr__`, and `__eq__`.

In [ ]:
from dataclasses import dataclass

@dataclass
class ClaimDC:
    claim_id: str
    reserve: float
    paid: float = 0.0          # typed field with a default

    def burn(self) -> float:
        return round(self.paid / self.reserve * 100, 1)

dc = ClaimDC("CLM-1", 5000.0, 1200.0)
print(dc)                      # auto __repr__
print("burn:", dc.burn(), "| equal to twin:", dc == ClaimDC("CLM-1", 5000.0, 1200.0))

## Decorators
A decorator wraps a function to add behavior (timing, logging, retry) without
changing the function's own code. This is the Advanced Track A5 material.

In [ ]:
import time
from functools import wraps

def timer(func):
    """Print how long a function took to run."""
    @wraps(func)                    # preserve func's name and docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"  {func.__name__!r} took {time.perf_counter() - start:.5f}s")
        return result
    return wrapper

@timer
def total_paid(rows):
    return sum(c["paid"] for c in rows)

print("total paid:", total_paid(claims))
print("name preserved:", total_paid.__name__)  # would print "wrapper" without @wraps

# PAUSE: comment out @wraps(func) and rerun. Watch total_paid.__name__ break to
# "wrapper". Put it back before moving on.

# Part 4: Put it together, then move to a script

## A small pipeline: aggregate and report by policy type

In [ ]:
summary = {}
for c in claims:
    bucket = summary.setdefault(c["policy_type"], {"reserve": 0.0, "paid": 0.0})
    bucket["reserve"] += c["reserve"]
    bucket["paid"] += c["paid"]

print("Loss ratio by policy type:")
for ptype in sorted(summary):
    b = summary[ptype]
    print(f"  {ptype}: {loss_ratio(b['paid'], b['reserve'])}%")

## This is Mini Project 1, in miniature
What you just watched, read the CSV, clean the numbers, aggregate by policy
type, compute a loss ratio, is exactly the Core and Challenge tiers of
`Activity_1_Mini_Capstone_Claims_Intake/` (**Mini Project 1**). Its Stretch
tier adds a second, nested JSON payments feed and flags reserve breaches for
the Special Investigations Unit. You have now seen every piece of it live.

Later today, `Activity_2_No_Show_Analysis/` (**Mini Project 2**) asks a
different kind of question: not just "what are the numbers," but "what do
they mean, and can you trust them." It reuses the same `csv` and aggregation
skills, aimed at interpretation instead of a pipeline. This demo does not
solve it live; that discovery is the point of the mini project.

## From notebook to script: why `.py` for your homework

Notebooks are for **exploring** cell by cell. Pipelines ship as `.py` files: they
run top to bottom, version-control cleanly, and are how your drills and homework
are structured. The same logic above lives in `demo_pipeline.py` in this folder,
wrapped in functions and a `main()`.

## How to run a `.py` file

From the folder that contains the script and its data:

```bash
uv run python demo_pipeline.py
```

`uv run` uses this project's `.venv`. The workflow you will use all course:

1. Work under `student-work/week2/day2/` so a `git pull` never conflicts.
2. The `.venv` lives in that day folder; run `uv` commands from there, and select
   it as your VS Code interpreter and notebook kernel.
3. Copy a drill's `Unsolved` scaffold and any `Resources/` into your project, then
   write the logic yourself.

Now open `Python Drills/` and work them as scripts, then the mini-capstones.